## 1. Configuración del archivo de logs
Todos los eventos y errores se guardan en `software_fj.log`. Es el **único** archivo que usa el sistema
(la información de clientes, servicios y reservas vive en objetos y listas).

In [ ]:
# --- Importaciones ---------------------------------------------------------
import logging                                   # Módulo estándar para registrar eventos en archivos
import re                                        # Expresiones regulares para validar correo y teléfono
from abc import ABC, abstractmethod              # Herramientas para crear clases abstractas
from datetime import datetime                    # Fecha y hora de creación de entidades
from functools import singledispatchmethod      # Permite "sobrecargar" métodos según el tipo del argumento

ARCHIVO_LOG = "software_fj.log"                  # Nombre del archivo donde se registran eventos y errores

logger = logging.getLogger("SoftwareFJ")         # Se crea (u obtiene) un logger con nombre propio
logger.setLevel(logging.DEBUG)                   # Se registran todos los niveles: DEBUG, INFO, WARNING, ERROR
logger.handlers.clear()                          # Evita handlers duplicados si la celda se ejecuta varias veces

manejador = logging.FileHandler(ARCHIVO_LOG, mode="w", encoding="utf-8")  # "w": log limpio en cada ejecución
formato = logging.Formatter("%(asctime)s | %(levelname)-8s | %(message)s",  # Formato: fecha | nivel | mensaje
                            datefmt="%Y-%m-%d %H:%M:%S")
manejador.setFormatter(formato)                  # Se asigna el formato al manejador de archivo
logger.addHandler(manejador)                     # Se conecta el manejador al logger

logger.info("===== Inicio del Sistema Software FJ =====")  # Primer evento registrado
print(f"Logger configurado. Los eventos se guardan en '{ARCHIVO_LOG}'.")

Logger configurado. Los eventos se guardan en 'software_fj.log'.


## 2. Excepciones personalizadas
Se crea una jerarquía propia: todas heredan de `SoftwareFJError`, así el sistema puede capturar
cualquier error del dominio con un solo `except` sin ocultar errores inesperados de Python.

In [ ]:
class SoftwareFJError(Exception):
    """Excepción base del sistema. Todas las demás heredan de ella."""
    def __init__(self, mensaje, codigo="FJ-000"):
        super().__init__(mensaje)                # Guarda el mensaje en la excepción estándar
        self.codigo = codigo                     # Código interno para identificar el tipo de error

    def __str__(self):
        return f"[{self.codigo}] {self.args[0]}" # Muestra el código junto al mensaje


class ClienteInvalidoError(SoftwareFJError):
    """Datos personales del cliente inválidos o faltantes."""
    def __init__(self, mensaje):
        super().__init__(mensaje, "FJ-100")


class ServicioInvalidoError(SoftwareFJError):
    """Servicio mal configurado (tarifa, capacidad, parámetros)."""
    def __init__(self, mensaje):
        super().__init__(mensaje, "FJ-200")


class ServicioNoDisponibleError(SoftwareFJError):
    """El servicio existe pero no está disponible para reservar."""
    def __init__(self, mensaje):
        super().__init__(mensaje, "FJ-210")


class ReservaInvalidaError(SoftwareFJError):
    """Intento de reserva con datos incorrectos."""
    def __init__(self, mensaje):
        super().__init__(mensaje, "FJ-300")


class OperacionNoPermitidaError(SoftwareFJError):
    """Operación prohibida según el estado actual (ej.: cancelar algo ya procesado)."""
    def __init__(self, mensaje):
        super().__init__(mensaje, "FJ-400")


class CalculoInconsistenteError(SoftwareFJError):
    """Un cálculo de costo dio un resultado ilógico (negativo, descuento > 100 %, etc.)."""
    def __init__(self, mensaje):
        super().__init__(mensaje, "FJ-500")


print("Excepciones personalizadas definidas:",
      [c.__name__ for c in SoftwareFJError.__subclasses__()])




Excepciones personalizadas definidas: ['ClienteInvalidoError', 'ServicioInvalidoError', 'ServicioNoDisponibleError', 'ReservaInvalidaError', 'OperacionNoPermitidaError', 'CalculoInconsistenteError']


## 3. Clase Abstracta EntidadesBase


In [ ]:

from abc import ABC, abstractmethod


class Entidad(ABC):

    def __init__(self, identificador, nombre):  
        self._identificador = identificador
        self._nombre = nombre

    @property
    def identificador(self):
        return self._identificador

    @property
    def nombre(self):
        return self._nombre

    @abstractmethod
    def validar(self):
        pass

    @abstractmethod
    def describir(self):
        pass


## 4. Clase Cliente que hereda de Entidad

In [ ]:

import re

from Entidades import Entidad
from Excepciones import ErrorValidacion


class Cliente(Entidad):

    def __init__(self, identificador, nombre, correo, telefono):

        super().__init__(identificador, nombre)

        # Atributos privados
        self.__correo = correo
        self.__telefono = telefono

        self.validar()

    @property
    def correo(self):
        return self.__correo

    @property
    def telefono(self):
        return self.__telefono

    def validar(self):

        if not str(self.identificador).strip():
            raise ErrorValidacion(
                "La identificacion no puede estar vacia"
            )

        if not isinstance(self.nombre, str) or not self.nombre.strip():
            raise ErrorValidacion(
                "El nombre no es valido"
            )

        patron_correo = r"^[\w.+-]+@[\w.-]+\.[a-zA-Z]{2,}$"

        if not isinstance(self.__correo, str) or not re.match(
            patron_correo, self.__correo
        ):
            raise ErrorValidacion(
                "El correo no tiene un formato valido"
            )

        if not str(self.__telefono).isdigit():
            raise ErrorValidacion(
                "El telefono debe contener solamente numeros"
            )

        if not 7 <= len(str(self.__telefono)) <= 15:
            raise ErrorValidacion(
                "El telefono debe tener entre 7 y 15 digitos"
            )

        return True

    def describir(self):

        return (
            f"Cliente: {self.nombre} | "
            f"Identificacion: {self.identificador} | "
            f"Correo: {self.__correo} | "
            f"Telefono: {self.__telefono}"
        )


# Prueba de funcionamiento
if __name__ == "__main__":

    try:
        cliente1 = Cliente(
            "1001",
            "Juan Perez",
            "juan@gmail.com",
            "3001234567"
        )

        print("CLIENTE REGISTRADO CORRECTAMENTE")
        print(cliente1.describir())

    except ErrorValidacion as error:
        print("ERROR AL REGISTRAR CLIENTE:", error)


## 5. Clase abstracta Servicio y servicios especializados

Esta sección implementa la clase abstracta `Servicio` y sus tres clases derivadas: `ServicioSala`, `ServicioEquipo` y `ServicioAsesoria`. Se reutilizan `Entidad`, las excepciones personalizadas y `logger` definidos en las secciones anteriores.

In [ ]:
# SECCIÓN 5. CLASE ABSTRACTA SERVICIO Y SERVICIOS ESPECIALIZADOS
import math
from abc import abstractmethod
from functools import singledispatchmethod


class Servicio(Entidad):
    #Clase abstracta comun para todos los servicios.

    """Hereda de Entidad para reutilizar identificador y nombre. Centraliza las
    validaciones generales y el cálculo de impuestos/descuentos. Las clases
    derivadas implementan sus propias reglas de validación y tarifa.
    """

    def __init__(self, identificador, nombre, tarifa_base, disponible=True):
        super().__init__(identificador, nombre)
        self.__tarifa_base = tarifa_base
        self.__disponible = disponible
        #La validación completa se ejecuta desde cada clase hija, despues de inicializar sus propios atributos

    @property
    def tarifa_base(self):
        #Devuelve la tarifa base sin permitir modificarla directamente
        return self.__tarifa_base

    @property
    def disponible(self):
        #Indica si el servicio está habilitado para recibir reservas
        return self.__disponible

    def validar(self):
        #Valida los atributos generales y delega los específicos a la hija
        if not isinstance(self.identificador, (str, int)) or not str(self.identificador).strip():
            mensaje = "El identificador del servicio no puede estar vacío."
            logger.error("Servicio inválido: %s", mensaje)
            raise ServicioInvalidoError(mensaje)

        if not isinstance(self.nombre, str) or not self.nombre.strip():
            mensaje = "El nombre del servicio debe ser un texto no vacío."
            logger.error("Servicio %s inválido: %s", self.identificador, mensaje)
            raise ServicioInvalidoError(mensaje)

        if (
            isinstance(self.__tarifa_base, bool)
            or not isinstance(self.__tarifa_base, (int, float))
            or not math.isfinite(self.__tarifa_base)
            or self.__tarifa_base <= 0
        ):
            mensaje = "La tarifa base debe ser un número positivo y finito."
            logger.error("Servicio %s inválido: %s", self.identificador, mensaje)
            raise ServicioInvalidoError(mensaje)

        if not isinstance(self.__disponible, bool):
            mensaje = "El atributo disponible debe ser True o False."
            logger.error("Servicio %s inválido: %s", self.identificador, mensaje)
            raise ServicioInvalidoError(mensaje)

        self._validar_configuracion()
        logger.info("Servicio validado correctamente: %s - %s", self.identificador, self.nombre)
        return True

    @abstractmethod
    def _validar_configuracion(self):
        #Valida los atributos particulares de cada servicio
        raise NotImplementedError

    def _validar_duracion(self, duracion):
        #Verifica que la duracion recibida sea numerica, finita y positiva
        if (
            isinstance(duracion, bool)
            or not isinstance(duracion, (int, float))
            or not math.isfinite(duracion)
            or duracion <= 0
        ):
            mensaje = "La duración debe ser un número positivo y finito (horas)."
            logger.error("Parámetro de servicio inválido: %s", mensaje)
            raise ServicioInvalidoError(mensaje)

        self.verificar_disponibilidad()
        return True

    def verificar_disponibilidad(self):
        #Impide utilizar un servicio que se encuentre deshabilitado
        if not self.__disponible:
            mensaje = f"El servicio '{self.nombre}' no está disponible para reservar."
            logger.warning(mensaje)
            raise ServicioNoDisponibleError(mensaje)
        return True

    def cambiar_disponibilidad(self, nuevo_estado):
        #Modifica el estado de disponibilidad mediante una operación controlada
        if not isinstance(nuevo_estado, bool):
            mensaje = "El nuevo estado de disponibilidad debe ser booleano."
            logger.error(mensaje)
            raise ServicioInvalidoError(mensaje)

        self.__disponible = nuevo_estado
        logger.info(
            "Disponibilidad del servicio %s modificada a %s",
            self.identificador,
            nuevo_estado,
        )

    def _validar_porcentaje(self, valor, nombre):
        #Valida impuesto o descuento expresado como decimal entre 0 y 1
        if (
            isinstance(valor, bool)
            or not isinstance(valor, (int, float))
            or not math.isfinite(valor)
            or not 0 <= valor <= 1
        ):
            mensaje = f"El {nombre} debe ser un número entre 0 y 1. Ejemplo: 0.19 equivale al 19 %."
            logger.error("Cálculo rechazado para el servicio %s: %s", self.identificador, mensaje)
            raise CalculoInconsistenteError(mensaje)

    @abstractmethod
    def validar_parametros(self, duracion, **opciones):
        #Cada servicio valida los parametros adicionales de su operación
        raise NotImplementedError

    @abstractmethod
    def _calcular_subtotal(self, duracion, **opciones):
        #Cada servicio implementa su propia formula de costo antes de impuestos
        raise NotImplementedError

    @singledispatchmethod
    def calcular_costo(self, datos, impuesto=0.19, descuento=0.0, **opciones):
        #Calculo generico: la implementación se selecciona según el tipo de datos
        mensaje = (
            "Formato no válido para calcular el costo. Use una duración numérica "
            "o un diccionario con la clave 'duracion'."
        )
        logger.error("Tipo de entrada no admitido en cálculo de servicio %s: %s", self.identificador, type(datos).__name__)
        raise ServicioInvalidoError(mensaje)

    @calcular_costo.register(int)
    @calcular_costo.register(float)
    def _(self, duracion, impuesto=0.19, descuento=0.0, **opciones):
        #Primera variante: recibe directamente la duracion como número
        try:
            self.validar_parametros(duracion, **opciones)
            self._validar_porcentaje(impuesto, "impuesto")
            self._validar_porcentaje(descuento, "descuento")
            subtotal = self._calcular_subtotal(duracion, **opciones)
        except TypeError as error:
            # Convierte errores de argumentos no reconocidos en un error del dominio,
            # conservando el error original mediante encadenamiento.
            logger.exception("Parámetros incompatibles al calcular el servicio %s", self.identificador)
            raise ServicioInvalidoError(
                "Se recibió un parámetro adicional no válido para este servicio."
            ) from error

        if (
            isinstance(subtotal, bool)
            or not isinstance(subtotal, (int, float))
            or not math.isfinite(subtotal)
            or subtotal < 0
        ):
            mensaje = "El subtotal calculado debe ser un número finito y no negativo."
            logger.error("Cálculo inconsistente en el servicio %s: %s", self.identificador, mensaje)
            raise CalculoInconsistenteError(mensaje)

        total = round(subtotal * (1 - descuento) * (1 + impuesto), 2)

        if not math.isfinite(total) or total < 0:
            mensaje = "El costo total calculado no es válido."
            logger.error("Cálculo inconsistente en el servicio %s: %s", self.identificador, mensaje)
            raise CalculoInconsistenteError(mensaje)

        logger.info(
            "Costo calculado | servicio=%s | subtotal=%.2f | impuesto=%.2f | descuento=%.2f | total=%.2f",
            self.identificador,
            subtotal,
            subtotal * (1 - descuento) * impuesto,
            subtotal * descuento,
            total,
        )
        return total

    @calcular_costo.register(dict)
    def _(self, datos, impuesto=0.19, descuento=0.0):
        """Segunda variante - recibe un diccionario de parametros
        Ej:
        servicio.calcular_costo({"duracion": 2, "personas": 5,
                                 "impuesto": 0.19, "descuento": 0.10})
        """
        parametros = datos.copy()

        try:
            duracion = parametros.pop("duracion")
        except KeyError as error:
            mensaje = "Falta la clave obligatoria 'duracion' en los datos del cálculo."
            logger.exception("Diccionario incompleto para el servicio %s", self.identificador)
            raise ServicioInvalidoError(mensaje) from error

        impuesto = parametros.pop("impuesto", impuesto)
        descuento = parametros.pop("descuento", descuento)

        # Se redirige a la variante numerica; las opciones restantes llegan a la validación especifica de cada tipo de servicio
        return self.calcular_costo(
            duracion,
            impuesto=impuesto,
            descuento=descuento,
            **parametros,
        )

    @abstractmethod
    def describir(self):
        #Devuelve una descripcion legible del servicio concreto
        raise NotImplementedError


class ServicioSala(Servicio):
    #Servicio de reserva de salas, con validacion de capacidad

    def __init__(self, identificador, nombre, tarifa_base, capacidad, disponible=True):
        self.__capacidad = capacidad
        super().__init__(identificador, nombre, tarifa_base, disponible)
        self.validar()

    @property
    def capacidad(self):
        return self.__capacidad

    def _validar_configuracion(self):
        if (
            isinstance(self.__capacidad, bool)
            or not isinstance(self.__capacidad, int)
            or self.__capacidad <= 0
        ):
            mensaje = "La capacidad de la sala debe ser un número entero mayor que cero."
            logger.error("Configuración inválida de sala %s: %s", self.identificador, mensaje)
            raise ServicioInvalidoError(mensaje)

    def validar_parametros(self, duracion, personas=1):
        self._validar_duracion(duracion)

        if isinstance(personas, bool) or not isinstance(personas, int) or personas <= 0:
            mensaje = "La cantidad de personas debe ser un entero mayor que cero."
            logger.error("Reserva de sala %s con parámetro inválido: %s", self.identificador, mensaje)
            raise ServicioInvalidoError(mensaje)

        if personas > self.__capacidad:
            mensaje = (
                f"La sala admite máximo {self.__capacidad} personas; "
                f"se solicitaron {personas}."
            )
            logger.warning("Capacidad insuficiente en sala %s: %s", self.identificador, mensaje)
            raise ServicioNoDisponibleError(mensaje)
        return True

    def _calcular_subtotal(self, duracion, personas=1):
        # La tarifa de sala se cobra por hora; las personas solo afectan capacidad
        return self.tarifa_base * duracion

    def describir(self):
        estado = "Disponible" if self.disponible else "No disponible"
        return (
            f"Sala | ID: {self.identificador} | Nombre: {self.nombre} | "
            f"Tarifa por hora: ${self.tarifa_base:,.2f} | "
            f"Capacidad: {self.__capacidad} personas | Estado: {estado}"
        )


class ServicioEquipo(Servicio):
    #Servicio de alquiler de equipos, cobrado por hora y por unidad

    def __init__(self, identificador, nombre, tarifa_base, cantidad_disponible, disponible=True):
        self.__cantidad_disponible = cantidad_disponible
        super().__init__(identificador, nombre, tarifa_base, disponible)
        self.validar()

    @property
    def cantidad_disponible(self):
        return self.__cantidad_disponible

    def _validar_configuracion(self):
        if (
            isinstance(self.__cantidad_disponible, bool)
            or not isinstance(self.__cantidad_disponible, int)
            or self.__cantidad_disponible < 0
        ):
            mensaje = "La cantidad disponible de equipos debe ser un entero igual o mayor que cero."
            logger.error("Configuración inválida de equipo %s: %s", self.identificador, mensaje)
            raise ServicioInvalidoError(mensaje)

    def validar_parametros(self, duracion, unidades=1):
        self._validar_duracion(duracion)

        if isinstance(unidades, bool) or not isinstance(unidades, int) or unidades <= 0:
            mensaje = "La cantidad de unidades a alquilar debe ser un entero mayor que cero."
            logger.error("Alquiler de equipo %s con parámetro inválido: %s", self.identificador, mensaje)
            raise ServicioInvalidoError(mensaje)

        if unidades > self.__cantidad_disponible:
            mensaje = (
                f"No hay suficientes unidades de '{self.nombre}'. "
                f"Disponibles: {self.__cantidad_disponible}; solicitadas: {unidades}."
            )
            logger.warning("Inventario insuficiente para equipo %s: %s", self.identificador, mensaje)
            raise ServicioNoDisponibleError(mensaje)
        return True

    def _calcular_subtotal(self, duracion, unidades=1):
        return self.tarifa_base * duracion * unidades

    def describir(self):
        estado = (
            "Disponible" if self.disponible and self.__cantidad_disponible > 0
            else "No disponible / sin existencias"
        )
        return (
            f"Alquiler de equipo | ID: {self.identificador} | Nombre: {self.nombre} | "
            f"Tarifa por hora/unidad: ${self.tarifa_base:,.2f} | "
            f"Unidades disponibles: {self.__cantidad_disponible} | Estado: {estado}"
        )


class ServicioAsesoria(Servicio):
    #Servicio de asesoria especializada, con modalidad virtual o presencial

    FACTOR_PRESENCIAL = 1.10  # Recargo ilustrativo del 10 % para modalidad presencial

    def __init__(self, identificador, nombre, tarifa_base, especialidad, disponible=True):
        self.__especialidad = especialidad
        super().__init__(identificador, nombre, tarifa_base, disponible)
        self.validar()

    @property
    def especialidad(self):
        return self.__especialidad

    def _validar_configuracion(self):
        if not isinstance(self.__especialidad, str) or not self.__especialidad.strip():
            mensaje = "La especialidad de la asesoría debe ser un texto no vacío."
            logger.error("Configuración inválida de asesoría %s: %s", self.identificador, mensaje)
            raise ServicioInvalidoError(mensaje)

    def validar_parametros(self, duracion, modalidad="virtual"):
        self._validar_duracion(duracion)

        if not isinstance(modalidad, str) or modalidad.lower() not in ("virtual", "presencial"):
            mensaje = "La modalidad debe ser 'virtual' o 'presencial'."
            logger.error("Asesoría %s con modalidad inválida: %s", self.identificador, modalidad)
            raise ServicioInvalidoError(mensaje)
        return True

    def _calcular_subtotal(self, duracion, modalidad="virtual"):
        subtotal = self.tarifa_base * duracion
        if modalidad.lower() == "presencial":
            subtotal *= self.FACTOR_PRESENCIAL
        return subtotal

    def describir(self):
        estado = "Disponible" if self.disponible else "No disponible"
        return (
            f"Asesoría especializada | ID: {self.identificador} | Nombre: {self.nombre} | "
            f"Especialidad: {self.__especialidad} | "
            f"Tarifa por hora: ${self.tarifa_base:,.2f} | Estado: {estado}"
        )
